# PCU-LAYER-PLACEMENT-001 — Layer-only diagnostic

Engineering-only causal diagnostic. Reuses the published L23 PCU-KILL-001 baseline and changes **only the target MoE decoder layer**.

New work: A-only, K=8, AdamW, LR=1e-3, 128 steps, batch=8 at deterministic early/mid MoE layers. L23 is not rerun. Formal seeds are never executed.

Requirements: Kaggle Internet ON, **T4 x2**, Secrets `HF_TOKEN` and `GITHUB_TOKEN`.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-composability-kill-001'
REPO = Path('/kaggle/working/mini-cells')
OUT = REPO / 'artifacts/research/pcu-layer-placement-001/engineering/26090501-layer-only'
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = '5.16.1'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 2, f'Need T4 x2; found {torch.cuda.device_count()} CUDA device(s)'
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'gpu0': torch.cuda.get_device_name(0),
    'gpu1': torch.cuda.get_device_name(1),
    'transformers': transformers.__version__,
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
assert hf_token and github_token
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GITHUB_TOKEN'] = github_token
login(token=hf_token, add_to_git_credential=False)
print('Secrets loaded; token values were not printed.')


In [ ]:
SEED_REGISTRY = REPO / 'research/formal_seed_registry.json'
def formal_states():
    payload = json.loads(SEED_REGISTRY.read_text())
    return {int(row['seed']): row['state'] for row in payload['seeds']}
expected = {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
assert formal_states() == expected
baseline = REPO / 'artifacts/research/pcu-kill-001/engineering/26090501-oracle-v2/ENGINEERING_DECISION.json'
assert baseline.is_file(), 'Published PCU-KILL-001 oracle-v2 baseline is required.'
baseline_decision = json.loads(baseline.read_text())
assert baseline_decision['status'] == 'LOCAL_CELL_MUTATION_UNSUPPORTED'
print(json.dumps({'baseline_status': baseline_decision['status'], 'formal_seed_states': formal_states()}, indent=2))


In [ ]:
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO / 'src')
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
run([sys.executable, '-m', 'pytest', '-q', 'tests/research/05-pcu-kill-001'], env=test_env)
run([sys.executable, '-m', 'compileall', '-q', 'src/minicells/pcu_kill_001', 'scripts/research'])
print('PCU layer-placement test/compile gate: PASS')


In [ ]:
existing = sorted(OUT.glob('LAYER_*.json')) if OUT.exists() else []
print(json.dumps({'resume': bool(existing), 'completed_layer_results': [p.name for p in existing]}, indent=2))
run([
    sys.executable, 'scripts/research/run_pcu_layer_placement_001.py',
    '--seed', '26090501',
    '--devices', 'cuda:0,cuda:1',
    '--out', OUT,
])


In [ ]:
required = ['RUN_IDENTITY.json', 'DESIGN.json', 'DECISION.json']
missing = [name for name in required if not (OUT / name).is_file()]
assert not missing, missing
layers = sorted(OUT.glob('LAYER_*.json'))
assert len(layers) == 2, [p.name for p in layers]
decision = json.loads((OUT / 'DECISION.json').read_text())
design = json.loads((OUT / 'DESIGN.json').read_text())
identity = json.loads((OUT / 'RUN_IDENTITY.json').read_text())
assert identity['source']['source_dirty'] is False
assert identity['formal_execution_not_started'] is True
assert decision['scientific_evidence'] is False
assert formal_states() == expected
print(json.dumps({
    'status': decision['status'],
    'targets': design['targets'],
    'comparison': decision['comparison'],
    'best': decision['best'],
    'formal_seed_states': formal_states(),
}, indent=2))


In [ ]:
run([sys.executable, 'scripts/research/publish_pcu_layer_placement_001.py', '--branch', BRANCH])
assert formal_states() == expected
print(json.dumps({'published': True, 'formal_seed_states': formal_states()}, indent=2))
